# VidTranscribe.ai - Colab Evaluation Pipeline

Notebook này chạy bộ benchmark mới nhất trên Google Colab:

1. Mount Google Drive
2. Clone/Pull repo
3. Cài FFmpeg, Ollama, Python dependencies
4. Kiểm tra dataset 100 câu: 70 PhoST + 30 ViMedCSS
5. Chạy Benchmark 3: Gemini 2.5 LLM Judge, 20 câu/batch, chạy song song theo batch
6. Chạy Performance benchmark
7. Chạy Group-level Sync benchmark
8. Copy kết quả về Google Drive


## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone/Pull Repository

In [3]:
import os
REPO_URL = 'https://github.com/HoangKhang226/VidTranscribe.ai.git'
REPO_DIR = '/content/VidTranscribe.ai'

if os.path.exists(REPO_DIR):
    %cd /content/VidTranscribe.ai
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd /content/VidTranscribe.ai

Cloning into '/content/VidTranscribe.ai'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 162 (delta 67), reused 136 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 177.43 KiB | 1.43 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/VidTranscribe.ai


## 3. Cài system dependencies: FFmpeg + Ollama

In [7]:
# 1. Cập nhật hệ thống, cài đặt FFmpeg VÀ zstd (bắt buộc cho Ollama)
!apt-get update -y && apt-get install -y ffmpeg zstd

# 2. Tải và cài đặt Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Khởi chạy Ollama background và ghi log
!nohup ollama serve > ollama.log 2>&1 &

# 4. Chờ 8 giây để server Ollama khởi động xong
import time
time.sleep(8)

# 5. Kéo model về (Đã xóa đoạn text thừa, hãy đảm bảo tên model "gemma4:e4b" là chính xác trên hệ thống của bạn)
!ollama pull gemma4:e4b

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
# import subprocess
# import time
# import os

# # 1. Buộc tắt toàn bộ các tiến trình Ollama cũ đang treo nếu có
# !pkill ollama

# # 2. Khởi chạy lại server Ollama ngầm
# print("Đang khởi chạy Ollama Server...")
# subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)

# # 3. Chờ server lên hẳn (tăng lên 12 giây cho chắc chắn)
# time.sleep(12)

# # 4. Kiểm tra xem server đã sống chưa
# result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
# print("Trạng thái Ollama:")
# print(result.stdout if result.returncode == 0 else "Server chưa phản hồi.")

## 4. Cài Python dependencies

In [5]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 12.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.7 MB/s eta 0:00:00
  Created wheel for distance: filename=Distance-0.1.3-py3-none-any.whl size=16256 sha256=432a0b0892c96b6291eca4e264b0854b4fc44d2190345c7a24f8d7b8e8b4d8fe
  Stored in directory: /root/.cache/pip/wheels/24/a8/58/407063d8e5c1d4dd6594c99d12baa0108570b5

## 5. Set Gemini API Key

In [6]:
import os, getpass
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Nhập GOOGLE_API_KEY cho Gemini judge: ')
print('GOOGLE_API_KEY đã được set:', bool(os.environ.get('GOOGLE_API_KEY')))

Nhập GOOGLE_API_KEY cho Gemini judge: ··········
GOOGLE_API_KEY đã được set: True


## 6. Kiểm tra / build dataset 100 câu
Repo đã có sẵn `evaluation/test_cases.json`. Cell này kiểm tra nếu thiếu hoặc không đủ 100 câu thì build lại từ PhoST local + ViMedCSS streaming text-only.

In [8]:
import json, os
cases_path = 'evaluation/test_cases.json'
need_build = True
if os.path.exists(cases_path):
    with open(cases_path, 'r', encoding='utf-8') as f:
        cases = json.load(f)
    print('Số case hiện có:', len(cases))
    need_build = len(cases) < 100

if need_build:
    !python evaluation/build_hybrid_dataset.py --phost-limit 70 --stress-limit 30 --output evaluation/test_cases.json

with open(cases_path, 'r', encoding='utf-8') as f:
    cases = json.load(f)
print('Dataset final:', len(cases), 'cases')
print('Sample keys:', cases[0].keys())

Số case hiện có: 100
Dataset final: 100 cases
Sample keys: dict_keys(['id', 'source', 'domain', 'type', 'input_raw', 'english_original', 'ai_translation', 'expected_localization', 'gold', 'predicted'])


## 7. Benchmark 3 - Gemini 2.5 LLM Judge

Cấu hình mặc định:
- 100 câu
- 20 câu / batch
- 5 batch song song (`--judge-concurrency 5`)
- structured output bằng LangChain `with_structured_output`

In [11]:
!python evaluation/benchmark_localization.py \
  --judge \
  --judge-model gemini-2.5-flash \
  --judge-batch-size 20 \
  --judge-concurrency 5

TEXT LOCALIZATION BENCHMARK
Cases            : /content/VidTranscribe.ai/evaluation/test_cases.json
Total cases       : 100
Avg TER           : 0.0000
Avg AI score      : 3.91/5
Target            : WARN
CSV output        : /content/VidTranscribe.ai/evaluation/localization_results.csv
JSON output       : /content/VidTranscribe.ai/evaluation/localization_results.json


## 8. Xem nhanh kết quả Localization Judge

In [12]:
import pandas as pd, json
df = pd.read_csv('evaluation/localization_results.csv')
display(df[['case_id', 'source', 'domain', 'english_original', 'ai_translation', 'expected_localization', 'ai_score', 'ai_reason']].head(20))
with open('evaluation/localization_results.json', 'r', encoding='utf-8') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

,case_id,source,domain,english_original,ai_translation,expected_localization,ai_score,ai_reason
0,phost_001,PhoST/local:1188.en+1188.vi,speech_flow,So I begin with an advertisement inspired by G...,"Vâng, Tôi bắt đầu với một quảng cáo được truyề...","Vâng, Tôi bắt đầu với một quảng cáo được truyề...",5.0,"Dịch chính xác, tự nhiên, phù hợp ngữ cảnh."
1,phost_002,PhoST/local:1188.en+1188.vi,speech_flow,"We are one people with one will, one resolve, ...","Chúng ta là một tập thể với một ý chí, một quy...","Chúng ta là một tập thể với một ý chí, một quy...",5.0,"Dịch chính xác, tự nhiên, phù hợp ngữ cảnh."
2,phost_003,PhoST/local:1188.en+1188.vi,speech_flow,"On January 24th, Apple Computer will introduce...","vào ngày 24 tháng 1, Hãng máy tính Apple sẽ gi...","vào ngày 24 tháng 1, Hãng máy tính Apple sẽ gi...",4.0,"Có lỗi chính tả nhỏ ở tên riêng ""Macitosh"" (th..."
3,phost_004,PhoST/local:1188.en+1188.vi,speech_flow,"And you'll see why 1984 wo n't be like ""1984.""",Và bạn sẽ thấy tại sao năm 1984 không hề giống...,Và bạn sẽ thấy tại sao năm 1984 không hề giống...,5.0,"Dịch chính xác, tự nhiên, phù hợp ngữ cảnh."
4,phost_005,PhoST/local:1188.en+1188.vi,speech_flow,So the underlying message of this video remain...,"Vâng, thông điệp cốt yếu của đoạn phim vẫn còn...","Vâng, thông điệp cốt yếu của đoạn phim vẫn còn...",5.0,"Dịch chính xác, tự nhiên, phù hợp ngữ cảnh."
5,phost_006,PhoST/local:1188.en+1188.vi,speech_flow,Technology created by innovative companies wil...,Công nghệ được tạo ra bởi những công ty cách t...,Công nghệ được tạo ra bởi những công ty cách t...,4.0,"Cụm từ ""tạo ra tự do cho chúng ta"" hơi cứng, c..."
6,phost_007,PhoST/local:1188.en+1188.vi,speech_flow,Fast - forward more than two decades: Apple la...,"Hơn hai thập kỷ sau đó, Apple ra mắt iPhone ở ...","Hơn hai thập kỷ sau đó, Apple ra mắt iPhone ở ...",3.0,"Cụm từ ""bỏ Đức Đạt Lai Lạt Ma ra"" và ""cho nhữn..."
7,phost_008,PhoST/local:1188.en+1188.vi,speech_flow,The American political cartoonist Mark Fiore a...,"Nhà vẽ tranh biếm họa chính trị Mỹ, Mark Fiore...","Nhà vẽ tranh biếm họa chính trị Mỹ, Mark Fiore...",4.0,"Cụm từ ""cũng có ứng dụng biếm họa của mình đượ..."
8,phost_009,PhoST/local:1188.en+1188.vi,speech_flow,His app was n't reinstated until he won the Pu...,Ứng dụng của ông đã không được chấp thuận cho ...,Ứng dụng của ông đã không được chấp thuận cho ...,2.0,"Dịch sai nghĩa từ ""reinstated"" thành ""chấp thu..."
9,phost_010,PhoST/local:1188.en+1188.vi,speech_flow,"The German magazine Stern, a news magazine, ha...","Một tạp chí tin tức của Đức, Stern, đã có nhữn...","Một tạp chí tin tức của Đức, Stern, đã có nhữn...",2.0,"Dịch sai nghĩa cụm ""Apple nannies"" và ""racy"". ..."


{
  "total_cases": 100,
  "avg_token_error_rate": 0.0,
  "exact_match_rate_percent": 100.0,
  "avg_ai_score": 3.91,
  "target_ter": 0.15,
  "target_judge_score": 4.5,
  "target_pass": false
}


## 9. Benchmark 1 - Performance & Hardware

In [24]:
!python evaluation/benchmark_perf.py --mode end_to_end --keep-temp-segments --include-nvidia-smi

END-TO-END PERFORMANCE & VRAM BENCHMARK
Video            : /content/VidTranscribe.ai/evaluation/Download.mp4
Video duration   : 95.11s
Model            : gemma4:e4b
Mode             : end_to_end
Hardsub          : False
PyTorch base VRAM : 0.00 MB
NVIDIA base VRAM  : 3 MB
07:58:36 - VidTranscribe - INFO - BẮT ĐẦU CHẠY PIPELINE (mode=end_to_end)...
07:58:36 - VidTranscribe - INFO - [Orchestrator - Khởi tạo] Memory Usage -> RAM: 788.21 MB | VRAM Allocated: 0.00 MB, Reserved: 0.00 MB
07:58:36 - VidTranscribe - INFO - Progress: 10% | Current Step: Bước 1: Ingestion (Tải/Phân tách Video & Audio)
07:58:36 - VidTranscribe - INFO - === BƯỚC 1: INGESTION ===
07:58:36 - VidTranscribe - INFO - Sử dụng file video: /content/VidTranscribe.ai/evaluation/Download.mp4
07:58:36 - VidTranscribe - INFO - Bắt đầu tách luồng từ video gốc: /content/VidTranscribe.ai/evaluation/Download.mp4
07:58:36 - VidTranscribe - INFO - [Ingestion - Khởi chạy FFmpeg] Memory Usage -> RAM: 788.23 MB | VRAM Allocated: 0.00 MB

## 10. Benchmark 2 - Group-level Sync

In [25]:
!python evaluation/benchmark_sync_group.py

GROUP-LEVEL AUDIO/VIDEO SYNC BENCHMARK
Group metadata   : /content/VidTranscribe.ai/src/output/audio/temp_segments/tts_groups.json
Total groups     : 16
------------------------------------------------------------------------
Valid groups     : 16
Missing audio    : 0
MAE overflow     : 43.75 ms
Max overflow     : 700 ms
Target           : PASS (MAE < 150ms)
CSV output       : /content/VidTranscribe.ai/evaluation/sync_group_results.csv
JSON output      : /content/VidTranscribe.ai/evaluation/sync_group_results.json


## 11. Copy kết quả về Google Drive

In [26]:
import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/VidTranscribe_Evaluation'
os.makedirs(DRIVE_DIR, exist_ok=True)

files_to_copy = [
    'evaluation/test_cases.json',
    'evaluation/localization_results.csv',
    'evaluation/localization_results.json',
    'evaluation/perf_results.json',
    'evaluation/sync_group_results.csv',
    'evaluation/sync_group_results.json',
    'evaluation/evaluation_results.md',
]

for file in files_to_copy:
    if os.path.exists(file):
        shutil.copy(file, DRIVE_DIR)
        print('Copied:', file)
    else:
        print('Missing:', file)

print('Done. Results saved to:', DRIVE_DIR)

Copied: evaluation/test_cases.json
Copied: evaluation/localization_results.csv
Copied: evaluation/localization_results.json
Copied: evaluation/perf_results.json
Copied: evaluation/sync_group_results.csv
Copied: evaluation/sync_group_results.json
Copied: evaluation/evaluation_results.md
Done. Results saved to: /content/drive/MyDrive/VidTranscribe_Evaluation
